In [1]:
%matplotlib inline

import matplotlib
matplotlib.rcParams['backend'] = 'module://matplotlib_inline.backend_inline'

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sp_signal

# Reproducibility
np.random.seed(42)

# Output directory
os.makedirs("outputs", exist_ok=True)

# Project constants
DATA_PATH = "201902020222_Data.csv"
FS = 100
SEG_LEN = 200
HOP = 25
N_AUG = 3

# Confirm backend
print("Matplotlib backend:", matplotlib.get_backend())
print("Environment setup complete")

Matplotlib backend: module://matplotlib_inline.backend_inline
Environment setup complete


In [2]:
print("=" * 65)
print("STEP 1: PREPROCESSING")
print("=" * 65)

print("\n[1/7] Loading raw data...")

df = pd.read_csv(DATA_PATH, sep='\t')
ppg_raw = df['ppg'].values.astype(float)

print(f"Rows loaded : {len(df):,}")
print(f"Columns     : {list(df.columns)}")
print(f"Sampling    : {FS} Hz")

STEP 1: PREPROCESSING

[1/7] Loading raw data...
Rows loaded : 14,401
Columns     : ['timestamp', 'ppg', 'hrm', 'accx', 'accy', 'accz', 'grax', 'gray', 'graz', 'gyrx', 'gyry', 'grz', 'pressure', 'stress']
Sampling    : 100 Hz


In [3]:
print("\n[2/7] Checking data quality...")

missing = df.isnull().sum()
has_missing = missing[missing > 0]

if len(has_missing) == 0:
    print("Missing values: 0")
else:
    for col, cnt in has_missing.items():
        print(f"{col}: {cnt}")

dupes = df.duplicated().sum()

print("Duplicates :", dupes)
print("Min :", ppg_raw.min())
print("Max :", ppg_raw.max())
print("Mean:", ppg_raw.mean())
print("Std :", ppg_raw.std())


[2/7] Checking data quality...
Missing values: 0
Duplicates : 0
Min : 1192206.0
Max : 3460640.0
Mean: 1584064.8870911743
Std : 407396.85051528725


In [4]:
print("\n[3/7] Applying Butterworth filter...")

sos = sp_signal.butter(
    4, [0.5, 8.0],
    btype='bandpass',
    fs=FS,
    output='sos'
)

ppg_filt = sp_signal.sosfiltfilt(sos, ppg_raw)

print("Filtered Std:", ppg_filt.std())


[3/7] Applying Butterworth filter...
Filtered Std: 54401.63182011409


In [5]:
print("\n[4/7] Applying smoothing...")

ppg_smooth = sp_signal.savgol_filter(
    ppg_filt,
    window_length=51,
    polyorder=3
)

print("After smoothing Std:", ppg_smooth.std())


[4/7] Applying smoothing...
After smoothing Std: 49305.969765280235


In [6]:
print("\n[5/7] Segmenting...")

clean_segs = []

for i in range(0, len(df) - SEG_LEN, HOP):
    seg = ppg_smooth[i:i + SEG_LEN]
    
    if len(seg) == SEG_LEN:
        clean_segs.append(seg)

clean_segs = np.array(clean_segs)

print("Segments:", len(clean_segs))


[5/7] Segmenting...
Segments: 569


In [7]:
def make_noisy(seg, seed=None):
    
    rng = np.random.RandomState(seed)
    std = np.std(seg)
    
    # Time vector for low-frequency variation
    t = np.linspace(0, 2*np.pi, len(seg))
    
    # Small amplitude fluctuations
    n = rng.normal(0, 0.25*std, len(seg))
    
    # Baseline drift simulation
    n += 0.08*std * np.sin(0.5*t + rng.uniform(0, np.pi))
    
    # Occasional transient disturbance
    if rng.random() < 0.4:
        pos = rng.randint(20, len(seg)-20)
        w = rng.randint(3, 8)
        n[pos:pos+w] += rng.choice([-1,1]) * 0.35*std * rng.randn(w)
    
    return seg + n


print("Controlled signal variation model ready")

Controlled signal variation model ready


In [8]:
print("\nApplying controlled signal variations for robustness evaluation...")

clean_aug = []
noisy_aug = []

for ci, seg in enumerate(clean_segs):
    for s in range(N_AUG):
        clean_aug.append(seg)
        noisy_aug.append(make_noisy(seg, seed=ci*10+s))

clean_aug = np.array(clean_aug)
noisy_aug = np.array(noisy_aug)

print(f"Original segments       : {len(clean_segs)}")
print(f"Processed segment pairs : {len(clean_aug)}")
print(f"Variations per segment  : {N_AUG}")


Applying controlled signal variations for robustness evaluation...
Original segments       : 569
Processed segment pairs : 1707
Variations per segment  : 3


In [10]:
print("\n[7/7] Normalizing segments (z-score)...")

# Mean and std per clean segment
mu_c = clean_aug.mean(axis=1, keepdims=True)
std_c = clean_aug.std(axis=1, keepdims=True) + 1e-8   # avoid divide by zero

# Normalize clean and noisy using clean stats
Xc = (clean_aug - mu_c) / std_c
Xn = (noisy_aug - mu_c) / std_c

print("Normalization complete")

# Sanity check
print(f"Xc mean : {Xc.mean():.6f} (≈ 0)")
print(f"Xc std  : {Xc.std():.6f}  (≈ 1)")
print(f"Xn mean : {Xn.mean():.6f}")
print(f"Xn std  : {Xn.std():.6f}")


[7/7] Normalizing segments (z-score)...
Normalization complete
Xc mean : 0.000000 (≈ 0)
Xc std  : 1.000000  (≈ 1)
Xn mean : -0.001107
Xn std  : 1.032893


In [11]:
print("\nSaving preprocessed outputs for Step 2...")

np.save("outputs/Xc.npy", Xc)
np.save("outputs/Xn.npy", Xn)

np.save("outputs/mu_c.npy", mu_c)
np.save("outputs/std_c.npy", std_c)

np.save("outputs/clean_segs.npy", clean_segs)
np.save("outputs/ppg_raw.npy", ppg_raw)
np.save("outputs/ppg_smooth.npy", ppg_smooth)

print("All Step 1 outputs saved successfully.")


Saving preprocessed outputs for Step 2...
All Step 1 outputs saved successfully.
